### Matrice de confucision

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Charger le meilleur modèle
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Rapport complet
print(classification_report(
    all_labels, all_preds,
    target_names=test_ds.classes
))

# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=test_ds.classes,
            yticklabels=test_ds.classes,
            cmap='Blues')
plt.title('Matrice de Confusion — Test Set')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/geoscanner/reports/confusion_matrix.png', dpi=150)
plt.show()
print("✅ Matrice sauvegardée dans Drive")

### Les courbes d'entrainements

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs('/content/drive/MyDrive/geoscanner/reports', exist_ok=True)

epochs_total = list(range(1, len(history['train_loss']) + 1))
phase_a_end  = 5  # nombre d'epochs Phase A

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Courbes d\'entraînement — GeoScanner ResNet-50', fontsize=14, fontweight='bold')

# ── Graphique 1 : Loss ──
ax1.plot(epochs_total, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax1.plot(epochs_total, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
ax1.axvline(x=phase_a_end, color='gray', linestyle='--', alpha=0.7, label='Phase A → B')
ax1.axvspan(1, phase_a_end, alpha=0.05, color='blue', label='Phase A')
ax1.axvspan(phase_a_end, len(epochs_total), alpha=0.05, color='orange', label='Phase B')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Train Loss vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Graphique 2 : Accuracy ──
ax2.plot(epochs_total, history['val_acc'], 'g-o', label='Val Accuracy', markersize=4)
ax2.axvline(x=phase_a_end, color='gray', linestyle='--', alpha=0.7, label='Phase A → B')
ax2.axvspan(1, phase_a_end, alpha=0.05, color='blue', label='Phase A')
ax2.axvspan(phase_a_end, len(epochs_total), alpha=0.05, color='orange', label='Phase B')
ax2.axhline(y=max(history['val_acc']), color='green', linestyle=':', alpha=0.7,
            label=f'Best: {max(history["val_acc"]):.2f}%')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/geoscanner/reports/training_curves.png', dpi=150)
plt.show()
print(f"✅ Courbes sauvegardées dans Drive")
print(f"✅ Meilleure Val Accuracy : {max(history['val_acc']):.2f}%")
print(f"✅ Epochs Phase A : 1 → {phase_a_end}")
print(f"✅ Epochs Phase B : {phase_a_end+1} → {len(epochs_total)}")